In [21]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
import re
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import os, tempfile
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from STICC_solver import STICC
%matplotlib inline
import matplotlib.pyplot as plt
import glob
from datetime import datetime
import umap.umap_ as umap
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

In [2]:
folder_path = Path("all_fires/")
parquet_files=[]
for file_path in folder_path.iterdir():
  if file_path.is_file():
    parquet_files.append(file_path.name)

In [9]:
features = [
    'viirs_m11', 'viirs_i2',
       'viirs_i1', 'ndvi', 'evi2', 'total_precipitation', 'wind_speed',
       'wind_direction', 'minimum_temperature', 'maximum_temperature',
       'energy_release_component', 'specific_humidity', 'slope', 'aspect',
       'elevation', 'pdsi', 'landcover_class', 'forecast_total_precipitation',
       'forecast_wind_speed', 'forecast_wind_direction',
       'forecast_temperature', 'forecast_specific_humidity'
]
def getFireDf(datestr):
    collections, fire_masks = [], []
    fire_id=datestr
    df = pd.read_parquet(f"all_fires/{fire_id}.parquet")
    # print(df.columns)
    dates = sorted(df["date"].unique())
    df = df[df["date"].isin(dates)].copy()
    df['date']=df['date'].apply(lambda x: datetime.strptime(x, '%Y-%m-%d').date())
    df["burning"] = df["active_fire"].notna().astype(float)
    df = df.sort_values(["x", "y", "date"])
    return df, dates
            
def plot_fire(df, dates):
    fire_df = df.copy()
    xmin, xmax = fire_df["x"].min(), fire_df["x"].max()
    ymin, ymax = fire_df["y"].min(), fire_df["y"].max()
    
    ncols = 4
    nrows = int(np.ceil(len(dates) / ncols))
    
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4*ncols, 4*nrows),
        sharex=True,
        sharey=True,
        squeeze=False
    )
    
    for ax, date in zip(axes.flat, dates):
    
        day = fire_df[fire_df["date"] == date]
    
        ax.scatter(
            day["x"],
            day["y"],
            c=day["active_fire"],
            cmap="bwr",
            vmin=0,
            vmax=1,
            marker="s",
            s=10
        )
    
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_aspect("equal", adjustable="box")
    
        ax.set_title(str(date))
    
    for ax in axes.flat[len(dates):]:
        ax.axis("off")
    
    return fig
    


def sw_dbscan(db, features, eps=.15, min_pts=4, d=1):
    db = db.copy().reset_index(drop=True)
    X = MinMaxScaler().fit_transform(db[features])
    labels = np.full(len(db), -1)
    visited = np.zeros(len(db), bool)
    cid = 0

    def neighbors(i):
        r, c = db.loc[i, ["row", "col"]]
        idx = np.where(
            (abs(db["row"].values - r) <= d) &
            (abs(db["col"].values - c) <= d)
        )[0]
        return idx[np.linalg.norm(X[idx] - X[i], axis=1) <= eps]

    for i in range(len(db)):
        if visited[i]: continue
        visited[i] = True
        N = neighbors(i)
        if len(N) < min_pts: continue

        labels[i] = cid
        Q = list(N)

        while Q:
            j = Q.pop()
            if not visited[j]:
                visited[j] = True
                Nj = neighbors(j)
                if len(Nj) >= min_pts: Q.extend(Nj)
            if labels[j] == -1: labels[j] = cid

        cid += 1

    db["cluster_id"] = labels
    return db   



def sticc_cluster(db, features, k=5, R=3, beta=3, lam=.01, max_iters=20):
    db = db.dropna(subset=features).copy().reset_index(drop=True)

    if len(db) < max(k, R + 1):
        db["cluster_id"] = -1
        return db

    X = StandardScaler().fit_transform(db[features])

    # R nearest spatial neighbors, excluding the cell itself
    R_eff = min(R, len(db) - 1)

    nn = NearestNeighbors(n_neighbors=R_eff)
    nn.fit(db[["row", "col"]])
    
    neighbors = nn.kneighbors(return_distance=False)

    data = np.hstack([
        np.arange(len(db))[:, None],
        X,
        neighbors
    ])

    with tempfile.NamedTemporaryFile(suffix=".txt", delete=False) as f:
        fname = f.name

    np.savetxt(fname, data, delimiter=",")

    os.makedirs("sticc_output", exist_ok=True)

    model = STICC(
        spatial_radius=R,
        number_of_clusters=k,
        lambda_parameter=lam,
        beta=beta,
        maxIters=max_iters,
        threshold=2e-5,
        write_out_file=False,
        prefix_string="sticc_output/",
        num_proc=1,
        biased=True,
        attr_idx_start=1,
        attr_idx_end=len(features),
        spatial_idx_start=len(features)+1,
        spatial_idx_end=len(features)+R
    )

    labels, mrfs = model.fit(input_file=fname)
    os.remove(fname)

    db["cluster_id"] = np.asarray(labels).astype(int)
    return db

def summarize_fire(df):
    data={
        "fire": df["fire"].iloc[0],
        "n_pixels": df[["row","col"]].drop_duplicates().shape[0],
        "duration_days":len(set(df["date"].to_list())),
        "actual fire days": df.loc[df["burning"] == 1, "date"].nunique(),
        "wind_mean": df["wind_speed"].mean(),
        "wind_std":  df["wind_speed"].std(),
        "temp_max_mean": df["maximum_temperature"].mean(),
        "temp_max_std":  df["maximum_temperature"].std(),
        "humidity_mean": df["specific_humidity"].mean(),
        "humidity_std":  df["specific_humidity"].std(),
        "erc_mean": df["energy_release_component"].mean(),
        "ndvi_mean": df["ndvi"].mean(),
        "slope_mean": df["slope"].mean(),
        "elevation_mean": df["elevation"].mean(),
    }
    return data

In [10]:
dfs=[]
for file in parquet_files:
    firename=file.strip(".parquet")
    df=getFireDf(firename)[0]
    dfs.append(summarize_fire(df))
summary_df= df = pd.DataFrame(dfs)
summary_df.head(2)

,fire,n_pixels,duration_days,actual fire days,wind_mean,wind_std,temp_max_mean,temp_max_std,humidity_mean,humidity_std,erc_mean,ndvi_mean,slope_mean,elevation_mean
0,fire_21748798,90350,12,5,4.569661,1.779441,301.686584,3.936381,0.002786,0.000481,94.393890,3032.699463,4.675699,1497.902954
1,fire_21889992,65751,17,4,3.285498,1.307124,306.394257,3.364468,0.006544,0.001019,71.360016,3797.102051,5.764729,604.603821


In [23]:
features = [
    'viirs_m11','viirs_i2','viirs_i1','evi2','total_precipitation',
    'wind_speed','wind_direction','minimum_temperature','maximum_temperature',
    'energy_release_component','specific_humidity','slope','aspect','elevation',
    'pdsi','landcover_class','forecast_total_precipitation','forecast_wind_speed',
    'forecast_wind_direction','forecast_temperature','forecast_specific_humidity'
]

for c in ["wind_direction", "forecast_wind_direction", "aspect"]:
    rad = np.deg2rad(df[c])
    df[c+"_sin"] = np.sin(rad)
    df[c+"_cos"] = np.cos(rad)

X = df[features]
y = df["burning"]

mask = y.notna()
X, y = X.loc[mask], y.loc[mask]

X_imp = SimpleImputer(strategy="median").fit_transform(X)
selector = SelectKBest(f_classif, k=10).fit(X_imp, y)

selected_features = [f for f, keep in zip(features, selector.get_support()) if keep]
selected_features

['viirs_m11',
 'viirs_i2',
 'wind_speed',
 'wind_direction',
 'maximum_temperature',
 'specific_humidity',
 'landcover_class',
 'forecast_wind_direction',
 'forecast_temperature',
 'forecast_specific_humidity']